In [ ]:
# ----------------------------
# Imports
# ----------------------------
import gc
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------
# Dataset
# ----------------------------
class HyperDataset(Dataset):
    def __init__(self, root, global_min, global_max):
        self.samples = []
        self.global_min = global_min
        self.scale = global_max - global_min
        root = Path(root)
        for class_dir in sorted(root.iterdir()):
            if not class_dir.is_dir():
                continue
            try:
                label = int(class_dir.name)
            except ValueError:
                continue
            for file in class_dir.glob("*.npy"):
                self.samples.append((file, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x = np.load(path).astype("float32")
        x = (x - self.global_min) / (self.scale + 1e-8)
        x = torch.from_numpy(x)
        y = torch.tensor(label, dtype=torch.long)
        return x, y

# ----------------------------
# Device
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------
# DataLoader
# ----------------------------
# data_dir = Path("../data/beyond-visible-spectrum-ai-for-agriculture-2025p2")
data_dir = Path("../data")
stats = np.load(data_dir / "processed" / "stats.npz")
global_min = float(stats["global_min"])
global_max = float(stats["global_max"])
print(f"Normalization: min={global_min}, max={global_max}")

# ----------------------------
# DataLoaders
# ----------------------------
BATCH_SIZE = 2

train_dataset = HyperDataset(data_dir / "Train", global_min, global_max)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

eval_dataset  = HyperDataset(data_dir / "evaluation", global_min, global_max)
eval_loader   = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Eval samples: {len(eval_dataset)}")

# ----------------------------
# Positional Encoding
# ----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

# ----------------------------
# Hybrid CNN + Transformer
# ----------------------------
class HybridCNNTransformer(nn.Module):
    def __init__(self, num_classes=10, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.1):
        super().__init__()
        # CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(125, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 128x128 -> 64x64
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64x64 -> 32x32
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)    # 32x32 -> 16x16
        )
        # Transformer
        self.seq_len = 16*16  # 256 tokens
        self.feature_dim = 256 # CNN channels
        self.pos_encoder = PositionalEncoding(self.feature_dim, max_len=self.seq_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.feature_dim, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # Classifier
        self.classifier = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        # x: (batch, 128,128,125) → (batch, channels, H, W)
        x = x.permute(0,3,1,2)  # (batch,125,128,128)
        x = self.cnn(x)         # (batch,256,16,16)
        batch, channels, H, W = x.shape
        x = x.flatten(2)        # (batch,256,256)
        x = x.permute(0,2,1)    # (batch, seq_len=256, feature_dim=256)
        x = self.pos_encoder(x)
        x = self.transformer(x)        # (batch, 256, 256)
        x = x.mean(dim=1)              # Global avg pooling -> (batch, 256)
        x = self.classifier(x)         # (batch, num_classes)
        return x

# ----------------------------
# Model, Loss, Optimizer
# ----------------------------
model = HybridCNNTransformer().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ----------------------------
# Training Loop
# ----------------------------
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc  = correct / total * 100
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

# ----------------------------
# Evaluation Loop
# ----------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in eval_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Evaluation Accuracy: {correct/total*100:.2f}%")

Using device: cpu
Normalization: min=0.0, max=28906.0
Train samples: 2089, Eval samples: 500
